# Phase 3: Text Detoxification (Mitigation)

In this final technical phase, we move beyond simply detecting toxicity. We aim to mitigate it by transforming toxic gaming comments into neutral, non-offensive statements while preserving the original core intent of the message.

**Objective:** Utilize a pre-trained Sequence-to-Sequence (Seq2Seq) model, specifically `s-nlp/bart-base-detox`, to rewrite flagged comments. This approach allows for a safer gaming environment without completely muting player communication.

In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

## 1. Model Initialization
We utilize the Hugging Face `pipeline` API for text-to-text generation. The `s-nlp/bart-base-detox` model is a BART architecture fine-tuned specifically for the task of parallel text detoxification.

In [3]:
print("Downloading and loading the BART Detoxification model and tokenizer...")

model_name = "s-nlp/bart-base-detox"

# Explicitly load the tokenizer and the Sequence-to-Sequence model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model and Tokenizer loaded successfully.")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully.


## 2. Testing on Gaming Jargon
To demonstrate the effectiveness of the model for our specific use case, we will test it on custom-crafted sentences that simulate highly toxic multiplayer gaming chat.

In [4]:
# A list of simulated highly toxic gaming comments
toxic_gaming_comments = [
    "you are such a stupid noob, uninstall the game right now you absolute trash",
    "stop feeding the enemy team you fucking idiot",
    "my teammates are completely brain dead garbage",
    "i swear to god i will kill you if you steal my loot again",
    "shut up and play the game you pathetic loser"
]

print("--- TEXT DETOXIFICATION RESULTS ---\n")

for comment in toxic_gaming_comments:
    print(f"Original Toxic : {comment}")
    
    # 1. Tokenize the input string
    inputs = tokenizer(comment, return_tensors="pt", truncation=True, max_length=128)
    
    # 2. Generate the detoxified output using the model
    outputs = model.generate(**inputs, max_new_tokens=50)
    
    # 3. Decode the output tokens back to a human-readable string
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Detoxified     : {result}")
    print("-" * 60)

--- TEXT DETOXIFICATION RESULTS ---

Original Toxic : you are such a stupid noob, uninstall the game right now you absolute trash
Detoxified     :  uninstall the game right now
------------------------------------------------------------
Original Toxic : stop feeding the enemy team you fucking idiot
Detoxified     : stop feeding the enemy team
------------------------------------------------------------
Original Toxic : my teammates are completely brain dead garbage
Detoxified     : My teammates are completely brain dead
------------------------------------------------------------
Original Toxic : i swear to god i will kill you if you steal my loot again
Detoxified     : i swear to god i will kill you if you steal my loot again
------------------------------------------------------------
Original Toxic : shut up and play the game you pathetic loser
Detoxified     : Don't play the game
------------------------------------------------------------


## 3. Quantitative Evaluation: Successful Toxicity Alleviation (STA)

While qualitative examples demonstrate the model's functionality, a robust evaluation requires quantitative metrics. We introduce the **Successful Toxicity Alleviation (STA)** score. 

To scientifically evaluate our mitigation pipeline, we dynamically sample 100 genuinely toxic comments directly from our dataset. These comments are processed through our BART detoxification model. The generated outputs are then evaluated by an independent, pre-trained RoBERTa toxicity classifier. 

The STA score represents the percentage of comments that successfully bypassed the toxicity filter (classified as neutral) after being rewritten.

In [3]:
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

print("1. Sampling 100 toxic comments from the dataset...")
# Load the dataset and filter for actual toxic comments
df_all = pd.read_csv('../data/train.csv')
df_toxic = df_all[df_all['toxic'] == 1].dropna(subset=['comment_text'])

# Sample 100 random toxic comments for a robust evaluation
toxic_sample = df_toxic.sample(n=100, random_state=42)
original_texts = toxic_sample['comment_text'].tolist()

print("1.5. Loading BART Model and Tokenizer...")
# Eksik olan tokenizer ve model tanımlamalarını buraya ekledik!
model_name = "s-nlp/bart-base-detox"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("2. Detoxifying comments using BART...")
detoxified_texts = []

for text in tqdm(original_texts, desc="Rewriting Texts"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    outputs = model.generate(**inputs, max_new_tokens=50)
    clean_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    detoxified_texts.append(clean_text)

print("3. Evaluating STA Score with pre-downloaded BERT...")
# we load the pre-downloaded 'unitary/toxic-bert' to act as the judge.
bert_evaluator = pipeline(
    "text-classification", 
    model="unitary/toxic-bert", 
    truncation=True, 
    max_length=512
)

successful_alleviations = 0
results_to_display = []

for orig, detox in tqdm(zip(original_texts, detoxified_texts), total=100, desc="Evaluating"):
    # The evaluator returns a label (e.g., 'toxic') and a score
    result = bert_evaluator(detox)[0]
    
    # If the output is classified as non-toxic (or toxic but score < 0.5), it was successful
    is_successful = result['score'] < 0.5 if result['label'] == 'toxic' else True
    
    if is_successful:
        successful_alleviations += 1
        
    # Save the first 3 examples for qualitative review
    if len(results_to_display) < 3:
        results_to_display.append((orig, detox, is_successful))

sta_score = successful_alleviations / 100

print("\n--- SUCCESSFUL TOXICITY ALLEVIATION (STA) RESULTS ---")
print(f"Total Toxic Comments Processed : 100")
print(f"Successfully Neutralized       : {successful_alleviations}")
print(f"STA Score                      : {sta_score:.2%}\n")

print("--- Sample Breakdown ---")
for orig, detox, success in results_to_display:
    status = "SUCCESS (Non-Toxic)" if success else "FAILED (Still Toxic)"
    print(f"[{status}]")
    print(f"Original : {orig}")
    print(f"Detox    : {detox}\n")

1. Sampling 100 toxic comments from the dataset...
1.5. Loading BART Model and Tokenizer...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

2. Detoxifying comments using BART...


Rewriting Texts:   0%|          | 0/100 [00:00<?, ?it/s]

3. Evaluating STA Score with pre-downloaded BERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]


--- SUCCESSFUL TOXICITY ALLEVIATION (STA) RESULTS ---
Total Toxic Comments Processed : 100
Successfully Neutralized       : 68
STA Score                      : 68.00%

--- Sample Breakdown ---
[FAILED (Still Toxic)]
Original : Hi! 

I wanna rape you!
Detox    : Hi! тАЬI wanna rape you!

[FAILED (Still Toxic)]
Original : Terrorize 
I will terrorise you for as long as you live - you are an arrogant, disgusting and rude excuse for a human being & you are going to die soon so I suggest you stop threatening people you bastard.
Detox    : I will terrorise you for as long as you live

[SUCCESS (Non-Toxic)]
Original : Being blocked

So that's your idea of mediation, is it? R Lopez can run amok through Wikipedia with his lying crap, and I get blocked for stating my intention of stopping him unless Wikipedia's rather creaky structures do so. I've seen some silly things since I have been editing here but that would be the silliest.
Detox    : Being blocked for stating my intention is not good.



## 4. Quantitative Evaluation: N-Gram Overlap (ROUGE-L)

A good detoxification model should act as a "minimal editor," removing only the toxic words while leaving the rest of the sentence intact, rather than completely rewriting the user's input from scratch.

To measure this, we use the **ROUGE-L** metric. ROUGE-L measures the longest common subsequence of words between the original and detoxified texts. A higher score indicates that the model successfully retained the original non-toxic vocabulary and sentence structure.

In [6]:
from rouge_score import rouge_scorer
import numpy as np

print("1. Initializing ROUGE Scorer...")
# We use ROUGE-L to measure the longest matching sequence of words
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

print("2. Calculating ROUGE-L scores for all comments...")
rouge_scores = []

for orig, detox in zip(original_texts, detoxified_texts):
    # Calculate the score between the original and detoxified text
    scores = scorer.score(orig, detox)
    # We extract the F1-measure of ROUGE-L
    rouge_scores.append(scores['rougeL'].fmeasure)

# Calculate the average
average_rouge = np.mean(rouge_scores)

print("\n--- N-GRAM OVERLAP (ROUGE-L) RESULTS ---")
print(f"Total Comments Evaluated : 100")
print(f"Average ROUGE-L Score    : {average_rouge:.4f} (Out of 1.0)")

# Find the examples with the most and least lexical overlap
max_rouge_idx = np.argmax(rouge_scores)
min_rouge_idx = np.argmin(rouge_scores)

print("\n--- Highest Lexical Retention (Max ROUGE-L) ---")
print(f"Score    : {rouge_scores[max_rouge_idx]:.4f}")
print(f"Original : {original_texts[max_rouge_idx]}")
print(f"Detox    : {detoxified_texts[max_rouge_idx]}")

print("\n--- Lowest Lexical Retention (Min ROUGE-L) ---")
print(f"Score    : {rouge_scores[min_rouge_idx]:.4f}")
print(f"Original : {original_texts[min_rouge_idx]}")
print(f"Detox    : {detoxified_texts[min_rouge_idx]}")

# Show two additional random examples for a broader qualitative check
random_indices = np.random.default_rng(42).choice(len(original_texts), size=2, replace=False)

print("\n--- Random Sample Breakdown ---")
for sample_number, idx in enumerate(random_indices, start=1):
    print(f"\n[Random Sample {sample_number}]")
    print(f"Score    : {rouge_scores[idx]:.4f}")
    print(f"Original : {original_texts[idx]}")
    print(f"Detox    : {detoxified_texts[idx]}")

1. Initializing ROUGE Scorer...
2. Calculating ROUGE-L scores for all comments...

--- N-GRAM OVERLAP (ROUGE-L) RESULTS ---
Total Comments Evaluated : 100
Average ROUGE-L Score    : 0.5874 (Out of 1.0)

--- Highest Lexical Retention (Max ROUGE-L) ---
Score    : 1.0000
Original : Hi! 

I wanna rape you!
Detox    : Hi! тАЬI wanna rape you!

--- Lowest Lexical Retention (Min ROUGE-L) ---
Score    : 0.0000
Original : )
Suck your dead mums toe- regards- Your Dad
Detox    : You should leave me alone.

--- Random Sample Breakdown ---

[Random Sample 1]
Score    : 0.6000
Original : Becouse no one gives a SHIT...
Detox    : Becouse no one cares

[Random Sample 2]
Score    : 0.0794
Original : "

Who the hell are you to decide who is notable, you ABSOLUTELY UNHEARD OF , BLITHERING NOBODY?

You invited me to edit a page and I did, so it is NOT ""DISRUPTION""!  If you don;t want it edited, DON'T MAKE IT EDITABLE!

My notability is verifiable on the Web, and off.  I cited STARRRING in a  American/Ca